# 03 - Tier-1 EdgeSpot Reproduction Colab

Notebook n?y t?ch ri?ng kh?i `02_train_enhanced.ipynb` ?? ch?y h??ng nghi?n c?u Tier-1:

- chu?n h?a benchmark EdgeSpot exact v?i true `_silence_`;
- train `EdgeSpotFull(tau=4)` thay v? DSCNN c?;
- d?ng `SCAF + GE2E` l?m baseline m?nh;
- t?y ch?n checkpoint selection b?ng `GSC-dev ACC@1% FAR`;
- ??nh gi? final b?ng `gsc_edgespot_exact` 100 trials.

Kh?ng ch?y notebook 02 trong c?ng workflow n?y. N?u ?ang d?ng A100, ch? c?n ch?y tu?n t? t? tr?n xu?ng.

## 0. Runtime Check

Ch?n `Runtime > Change runtime type > A100 GPU` tr??c khi ch?y. Cell n?y ch? ki?m tra GPU v? disk.

In [ ]:
import os, shutil, subprocess, sys, json, yaml, time
from pathlib import Path

print('Python:', sys.version)
free_gb = shutil.disk_usage('/content').free / 1024**3
print(f'Free disk /content: {free_gb:.1f} GB')

try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch not ready yet:', exc)

## 1. Config Chung

??y l? cell duy nh?t th??ng c?n s?a. M?c ??nh d?ng MSWC Top500 full/unlimited clips ?? b?m paper-grade h?n; n?u ch? debug nhanh m?i ??i v? mpw200.

In [ ]:
# === Project / Drive ===
GITHUB_REPO = 'https://github.com/AnHgPham/DoAnTotNghiep.git'
PROJECT_DIR = Path('/content/DoAnTotNghiep')
DRIVE_PROJECT = '/content/drive/MyDrive/DoAnTotNghiep_output'

# === Data policy ===
# Paper-grade: top500 + unlimited clips per word.
# Fast debug only: set MSWC_MAX_PER_WORD = 200.
# Full English is much larger: set MSWC_SPLIT_MODE = 'full', MSWC_MAX_PER_WORD = 0.
MSWC_SPLIT_MODE = 'top500'      # top500 | full
MSWC_MAX_PER_WORD = 0           # 0 = unlimited/full for paper-grade; 200 = fast debug only
MIN_CACHE_COVERAGE = 0.90

# === Tier-1 model/run ===
MODEL_FAMILY = 'edgespot_full'  # edgespot_full | bcresnet_fs | edgespot_lite | dscnn
EDGE_TAU = 4
LOSS_NAME = 'scaf_ge2e'         # scaf | ge2e | scaf_ge2e | kd_scaf_ge2e
CAP_LABEL = 'full' if int(MSWC_MAX_PER_WORD) <= 0 else f'mpw{MSWC_MAX_PER_WORD}'
RUN_TAG = f'{MODEL_FAMILY}_t{EDGE_TAU}_{LOSS_NAME}_{MSWC_SPLIT_MODE}_{CAP_LABEL}'

# === Training scale ===
MAIN_EPOCHS = 40
MAIN_EPISODES = 600
NUM_WORKERS = 2

# GSC-dev selection is scientifically cleaner but slower.
SELECT_BY_GSC_DEV = True
GSC_DEV_EVERY = 2
GSC_DEV_RUNS = 5
GSC_DEV_K_SHOT = 10

print('RUN_TAG:', RUN_TAG)
print('Drive project:', DRIVE_PROJECT)

## 2. Mount Drive + Sync Code

Cell n?y an to?n khi rerun: n?u repo ?? t?n t?i th? ch? update code b?ng git, kh?ng x?a `data/` ?? t?i.

In [ ]:
from google.colab import drive
import os, subprocess
from pathlib import Path

drive.mount('/content/drive')
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/results', exist_ok=True)

%cd /content
if not (PROJECT_DIR / '.git').exists():
    !git clone {GITHUB_REPO} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git fetch origin
    !git reset --hard origin/main

%cd {PROJECT_DIR}
!git log -1 --oneline

## 3. Install Dependencies

N?u Colab y?u c?u restart runtime sau khi c?i package l?n, restart r?i ch?y l?i t? cell 0 ??n ??y.

In [ ]:
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Smoke Test Source Code

B?t l?i model/protocol tr??c khi t?i ho?c train l?u.

In [ ]:
%cd {PROJECT_DIR}
!python -m pytest tests/test_edgespot_full.py tests/test_ge2e.py tests/test_gsc_silence_provider.py -q
!python scripts/model_report.py --family edgespot_full --tau 4

## 5. Prepare GSC v2

GSC d?ng cho evaluation v? GSC-dev checkpoint selection. N?u ?? c? r?i th? cell s? skip.

In [ ]:
%cd {PROJECT_DIR}
from pathlib import Path

if not Path('data/gsc_v2/testing_list.txt').exists():
    !python data/download_gsc.py
else:
    wav_n = len(list(Path('data/gsc_v2').rglob('*.wav')))
    print(f'GSC already exists: {wav_n} WAV files')

## 6. Prepare MSWC From Google Drive Cache

Cache hit s? d?ng WAV tr?n Drive, kh?ng t?i/convert l?i. V?i `MSWC_MAX_PER_WORD = 0`, cache name l? `mswc_en_wav_top500_full`; l?n ??u s? ph?i t?i/extract/convert l?n h?n nhi?u so v?i mpw200.

In [ ]:
%cd {PROJECT_DIR}
import os, shutil, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', force=True)

from data.mswc_drive_cache import setup_mswc_from_drive, drive_cache_status, cache_dir_name

print('MSWC cache:', cache_dir_name(MSWC_SPLIT_MODE, MSWC_MAX_PER_WORD))
print('Policy:', MSWC_SPLIT_MODE, 'max_per_word=', MSWC_MAX_PER_WORD)

_from_drive_cache = setup_mswc_from_drive(
    DRIVE_PROJECT,
    split_mode=MSWC_SPLIT_MODE,
    max_per_word=MSWC_MAX_PER_WORD,
    min_train_val_coverage=MIN_CACHE_COVERAGE,
    n_cpu=os.cpu_count() or 8,
)
print('Loaded from Drive cache:', _from_drive_cache)

status = drive_cache_status(DRIVE_PROJECT, MSWC_SPLIT_MODE, MSWC_MAX_PER_WORD)
print('Drive cache status:', {k: status[k] for k in [
    'n_wav', 'n_word_dirs', 'required_present', 'required_total', 'required_coverage'
]})

## 7. Prepare DEMAND Noise

N?u DEMAND ?? ?? 272 WAV th? cell skip. N?u b? l?i 504 gi?a ch?ng, cell t? x?a cache d? v? t?i l?i.

In [ ]:
%cd {PROJECT_DIR}
import shutil, time, zipfile, requests
from pathlib import Path
from tqdm import tqdm

DEMAND_DIR = Path('data/demand')
MIN_DEMAND_WAV = 250
BASE = 'https://zenodo.org/records/1227121/files'
ENVS = [
    'DKITCHEN','DLIVING','DWASHING','NFIELD','NPARK','NRIVER',
    'OHALLWAY','OMEETING','OOFFICE','PCAFETER','PRESTO','PSTATION',
    'SPSQUARE','STRAFFIC','TBUS','TCAR','TMETRO',
]

def _download_file(url: str, out_path: Path, retries: int = 3):
    tmp_path = out_path.with_suffix(out_path.suffix + '.part')
    for attempt in range(1, retries + 1):
        try:
            if tmp_path.exists():
                tmp_path.unlink()
            with requests.get(url, stream=True, timeout=(20, 180)) as r:
                r.raise_for_status()
                with open(tmp_path, 'wb') as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
            tmp_path.replace(out_path)
            return
        except Exception as exc:
            if tmp_path.exists():
                tmp_path.unlink()
            if attempt == retries:
                raise
            wait = 5 * attempt
            print(f'Retry {attempt}/{retries} after {type(exc).__name__}: {exc}. Waiting {wait}s...')
            time.sleep(wait)

DEMAND_DIR.mkdir(parents=True, exist_ok=True)
existing = len(list(DEMAND_DIR.rglob('*.wav')))
if 0 < existing < MIN_DEMAND_WAV:
    print(f'DEMAND incomplete: {existing} WAV < {MIN_DEMAND_WAV}. Re-downloading cleanly...')
    shutil.rmtree(DEMAND_DIR, ignore_errors=True)
    DEMAND_DIR.mkdir(parents=True, exist_ok=True)
    for stale_zip in Path('data').glob('*_16k.zip*'):
        stale_zip.unlink(missing_ok=True)

if len(list(DEMAND_DIR.rglob('*.wav'))) < MIN_DEMAND_WAV:
    for env in tqdm(ENVS, desc='Downloading DEMAND'):
        url = f'{BASE}/{env}_16k.zip?download=1'
        zp = Path(f'data/{env}_16k.zip')
        if not zp.exists():
            _download_file(url, zp)
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(DEMAND_DIR)
        zp.unlink(missing_ok=True)

print('DEMAND WAV:', len(list(DEMAND_DIR.rglob('*.wav'))))

## 8. Verify Data + Build Tier-1 Config

Cell n?y t?o `/content/tier1_colab.yaml` ?? checkpoint/result l?u v?o Drive v? d?ng ??ng data hi?n t?i.

In [ ]:
%cd {PROJECT_DIR}
import yaml, json, os
from pathlib import Path

for d in ['data/gsc_v2', 'data/mswc_en/clips', 'data/demand']:
    p = Path(d)
    if p.exists():
        wav_n = len(list(p.rglob('*.wav')))
        print(f'{d}: {wav_n} WAV')
    else:
        print(f'MISSING: {d}')

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['checkpoint']['dir'] = f'{DRIVE_PROJECT}/checkpoints'
cfg['data']['mswc_dir'] = 'data/mswc_en'
cfg['data']['train_dir'] = 'data/mswc_en'
cfg['data']['gsc_dir'] = 'data/gsc_v2'
cfg['data']['demand_dir'] = 'data/demand'
cfg['noise']['demand_dir'] = 'data/demand'
cfg['data']['max_per_word'] = 0
cfg['model']['family'] = MODEL_FAMILY
cfg['model']['edge_width_mult'] = EDGE_TAU

with open('/content/tier1_colab.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

print('Saved config: /content/tier1_colab.yaml')
print('Checkpoint dir:', cfg['checkpoint']['dir'])

## 9. Smoke Train Nhanh

B?t bu?c ch?y tr??c train th?t. N?u cell n?y l?i th? kh?ng ch?y cell 10.

In [ ]:
%cd {PROJECT_DIR}
!python scripts/train.py \
  --config /content/tier1_colab.yaml \
  --model-family edgespot_full \
  --edge-tau 1 \
  --loss scaf_ge2e \
  --run-tag smoke_edgespot_t1 \
  --epochs 1 \
  --episodes 5 \
  --max-per-word 20 \
  --num-workers 2

## 10. Train Tier-1 Ch?nh: EdgeSpotFull tau=4 + SCAF+GE2E

??y l? cell train th?t. V?i A100 v? paper-grade data, b?t ??u b?ng Top500 full tr??c. N?u mu?n ch?y nhanh ?? ki?m tra pipeline, gi?m `MAIN_EPOCHS` ho?c `MAIN_EPISODES` ? cell config, ho?c ??i `MSWC_MAX_PER_WORD = 200` ?? debug.

In [ ]:
%cd {PROJECT_DIR}

select_args = ''
if SELECT_BY_GSC_DEV:
    select_args = f'--select-by-gsc-dev --gsc-dev-every {GSC_DEV_EVERY} --gsc-dev-runs {GSC_DEV_RUNS} --gsc-dev-k-shot {GSC_DEV_K_SHOT}'

cmd = f'''python scripts/train.py \
  --config /content/tier1_colab.yaml \
  --model-family {MODEL_FAMILY} \
  --edge-tau {EDGE_TAU} \
  --loss {LOSS_NAME} \
  --run-tag {RUN_TAG} \
  --epochs {MAIN_EPOCHS} \
  --episodes {MAIN_EPISODES} \
  --num-workers {NUM_WORKERS} \
  {select_args}'''
print(cmd)
!{cmd}

## 11. Resume Train N?u Colab B? Ng?t

Ch? ch?y cell n?y n?u cell train ch?nh b? ng?t gi?a ch?ng. N? resume t? `latest.pt` c?ng `RUN_TAG`.

In [ ]:
%cd {PROJECT_DIR}
from pathlib import Path
latest = Path(DRIVE_PROJECT) / 'checkpoints' / RUN_TAG / 'latest.pt'
print('Latest checkpoint:', latest, latest.exists())

select_args = ''
if SELECT_BY_GSC_DEV:
    select_args = f'--select-by-gsc-dev --gsc-dev-every {GSC_DEV_EVERY} --gsc-dev-runs {GSC_DEV_RUNS} --gsc-dev-k-shot {GSC_DEV_K_SHOT}'

if latest.exists():
    cmd = f'''python scripts/train.py \
      --config /content/tier1_colab.yaml \
      --model-family {MODEL_FAMILY} \
      --edge-tau {EDGE_TAU} \
      --loss {LOSS_NAME} \
      --run-tag {RUN_TAG} \
      --epochs {MAIN_EPOCHS} \
      --episodes {MAIN_EPISODES} \
      --num-workers {NUM_WORKERS} \
      --resume {latest} \
      {select_args}'''
    print(cmd)
    !{cmd}
else:
    print('No latest.pt found. Run main training first.')

## 12. Evaluate EdgeSpot Exact 100 Trials

Ch?y sau khi c? `best.pt`. ??y l? benchmark ch?nh ?? so v?i EdgeSpot.

In [ ]:
%cd {PROJECT_DIR}
from pathlib import Path
best = Path(DRIVE_PROJECT) / 'checkpoints' / RUN_TAG / 'best.pt'
out_dir = Path(DRIVE_PROJECT) / 'results' / RUN_TAG / 'edgespot_exact'
print('Best checkpoint:', best, best.exists())
print('Output:', out_dir)
assert best.exists(), 'Chua co best.pt. Hay train xong truoc.'

!python scripts/evaluate_edgespot_protocol.py \
  --checkpoint {best} \
  --model-family {MODEL_FAMILY} \
  --edge-tau {EDGE_TAU} \
  --k-shot 10 \
  --n-runs 100 \
  --gsc-query-split test \
  --output-dir {out_dir}

## 13. Research Table

In b?ng metric ?? copy v?o b?o c?o.

In [ ]:
%cd {PROJECT_DIR}
from pathlib import Path
out_dir = Path(DRIVE_PROJECT) / 'results' / RUN_TAG / 'edgespot_exact'
jsons = sorted(out_dir.glob('*_results.json'))
print('Result JSON:', [str(p) for p in jsons])
if jsons:
    !python scripts/make_research_tables.py {' '.join(str(p) for p in jsons)}
else:
    print('No result JSON found yet.')

## 14. Optional KD Phase Sau

Kh?ng ch?y ngay n?u SCAF+GE2E ch?a ?n. KD c?n teacher projection head t?t; n?u d?ng head random th? ch? l? smoke test, kh?ng claim k?t qu? khoa h?c.

In [ ]:
# Optional: precompute teacher embeddings. This is intentionally commented out.
# %cd {PROJECT_DIR}
# !python scripts/precompute_teacher_embeddings.py \
#   --data-dir data/mswc_en \
#   --split train \
#   --output-dir {DRIVE_PROJECT}/teacher_w2v2_train \
#   --batch-size 16
#
# !python scripts/train.py \
#   --config /content/tier1_colab.yaml \
#   --model-family edgespot_full \
#   --edge-tau 4 \
#   --loss kd_scaf_ge2e \
#   --teacher-embeddings-dir {DRIVE_PROJECT}/teacher_w2v2_train \
#   --run-tag edgespot_full_t4_kd_scaf_ge2e \
#   --epochs 40 \
#   --episodes 600 \
#   --num-workers 2
print('KD phase is optional and disabled by default.')